In [8]:
import os 
import sys
import torch 
import torch.nn as nn 
from torch.utils.data import DataLoader , random_split

import matplotlib.pyplot as plt
sys.path.append('..')

from src.dataset import HiveAudioDataset
from src.model import HiveCNN


In [9]:
raw_path = "../data/raw"
classes = {'QueenBee Absent':0 , 'QueenBee Present':1}

file_paths = []
labels = []

for class_name , label in classes.items():
    folder = os.path.join(raw_path , class_name)
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.endswith('wav'):
                file_paths.append(os.path.join(folder , f))
                labels.append(label)

print(f'Total Audio Files Found : {len(file_paths)}')

print(f'files are \n {file_paths[:5]}')


Total Audio Files Found : 6000
files are 
 ['../data/raw\\QueenBee Absent\\Hive1 31_05_2018_NO_QueenBee____00_00_00_chunk1.wav', '../data/raw\\QueenBee Absent\\Hive1 31_05_2018_NO_QueenBee____00_00_00_chunk10.wav', '../data/raw\\QueenBee Absent\\Hive1 31_05_2018_NO_QueenBee____00_00_00_chunk11.wav', '../data/raw\\QueenBee Absent\\Hive1 31_05_2018_NO_QueenBee____00_00_00_chunk12.wav', '../data/raw\\QueenBee Absent\\Hive1 31_05_2018_NO_QueenBee____00_00_00_chunk13.wav']


In [ ]:
# instantiating Pytorch Dataset
full_dataset = HiveAudioDataset(file_path=file_paths,labels=labels , duration=3.0)

print(len(full_dataset))


# splitting the Data
size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - size 


train_dataset , valid_dataset = random_split(full_dataset , [train_size , size], 
                                             generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset , batch_size = 32, shuffle = True , num_workers= 0 )
val_loader = DataLoader(valid_dataset , batch_size = 32 , shuffle = False, num_workers = 0)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

6000
Train batches: 150, Val batches: 38


In [11]:
#initializing the model 
model = HiveCNN()
learning_rate = 1e-3 #0.001
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters() , lr = learning_rate , weight_decay= 1e-4)
print(model)

HiveCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=Tru

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [13]:
# training and validation loop 

epochs = 10 
history = {'train_loss':[] , 'val_loss':[] , 'val_acc':[]}


for e in range(epochs):
    model.train()
    running_loss = 0.0
    
    for i , t in train_loader:
        i , t = i.to(device) , t.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        
        outputs = model(i)
        loss = loss_fn(outputs , t)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * i.size(0)
    e_train_loss = running_loss/ len(train_dataset)
    
    #validation phase 
    model.eval()
    
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for i , t in val_loader:
             i , t = i.to(device)  , t.to(device).unsqueeze(1)
             
             optimizer.zero_grad()
             outputs = model(i)
             loss = loss_fn(outputs , t)
             val_loss += loss.item() * i.size(0)
             
             preds = (torch.sigmoid(outputs) > 0.5).float()
             correct += (preds == t).sum().item()
             total += t.size(0)
             
    epochs_val_loss = val_loss / len(valid_dataset)
    epochs_val_accuracy = (correct/total)*100
    
    
    history['train_loss'].append(e_train_loss)
    history['val_loss'].append(epochs_val_loss)
    history['val_acc'].append(epochs_val_accuracy)
    
    print(f"Epoch [{e+1}/{epochs}] | Train Loss: {e_train_loss:.4f} | Val Loss: {epochs_val_loss:.4f} | Val Acc: {epochs_val_accuracy:.2f}%")

Epoch [1/10] | Train Loss: 0.1024 | Val Loss: 0.0142 | Val Acc: 99.67%
Epoch [2/10] | Train Loss: 0.0189 | Val Loss: 0.0081 | Val Acc: 99.67%
Epoch [3/10] | Train Loss: 0.0175 | Val Loss: 0.0036 | Val Acc: 99.92%
Epoch [4/10] | Train Loss: 0.0063 | Val Loss: 0.0258 | Val Acc: 99.17%
Epoch [5/10] | Train Loss: 0.0046 | Val Loss: 0.0018 | Val Acc: 99.92%
Epoch [6/10] | Train Loss: 0.0037 | Val Loss: 0.0004 | Val Acc: 100.00%
Epoch [7/10] | Train Loss: 0.0030 | Val Loss: 2.2813 | Val Acc: 66.75%
Epoch [8/10] | Train Loss: 0.0073 | Val Loss: 0.0166 | Val Acc: 99.58%
Epoch [9/10] | Train Loss: 0.0081 | Val Loss: 0.0016 | Val Acc: 100.00%
Epoch [10/10] | Train Loss: 0.0049 | Val Loss: 0.0032 | Val Acc: 99.92%
